# L01 · 环境配置 — conda/venv、Jupyter 内核选择、`import aurora` 一次跑通

**本课目标**：确认 AURORA 的 Python 环境 (Python environment)、虚拟环境 (virtual environment)、Jupyter 内核 (Jupyter kernel)、项目路径 (project path) 和核心依赖 (core dependencies) 都已经连在一起。

完成本课后，再进入 `f1_notebook_python_numpy.ipynb`。后面的数学和 Audio Core Notebook 都默认这里已经通过。

**本课产出**：一份可复制的环境报告 (environment report)，包括 Python 路径、项目根目录、包导入结果和内核选择结果。


## 0. 先在终端完成一次安装 (terminal setup)

在终端 (terminal) 里运行下面命令。它们只需要做一次。

```bash
cd /Users/z/AURORA
python -m venv .venv
source .venv/bin/activate
python -m pip install -U pip
python -m pip install -e ".[dev,notebooks]"
python -m ipykernel install --user --name aurora --display-name "Python (AURORA)"
jupyter lab
```

每行命令的作用：

| 命令 | 作用 |
|---|---|
| `python -m venv .venv` | 创建虚拟环境 (virtual environment) |
| `source .venv/bin/activate` | 激活虚拟环境 (activate environment) |
| `python -m pip install -e ".[dev,notebooks]"` | 用可编辑安装 (editable install) 安装 AURORA 和 Notebook 依赖 |
| `python -m ipykernel install ...` | 注册 Jupyter 内核 (register Jupyter kernel) |
| `jupyter lab` | 启动 JupyterLab (JupyterLab) |


## 1. 选择正确内核 (select the correct kernel)

打开本 Notebook 后，看右上角内核名称。

请选：`Python (AURORA)`。

如果右上角不是这个名字：

1. 点右上角内核名称。
2. 选择 `Python (AURORA)`。
3. 重新运行下面的检查格。

内核 (kernel) 决定 Notebook 用哪个 Python 解释器 (Python interpreter)。同一台电脑上可能有多个 Python；本课要确认 Notebook 使用的是 AURORA 的虚拟环境。


In [ ]:
import sys
import platform
from pathlib import Path

print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Current working directory:', Path.cwd())


## 2. 读懂 `sys.executable` (Python executable)

`sys.executable` 是当前内核正在使用的 Python 程序路径。

常见情况：

| 路径特征 | 判断 |
|---|---|
| 包含 `/Users/z/AURORA/.venv/` | 正在使用 AURORA 虚拟环境 |
| 包含 `conda` 或 `miniconda` | 正在使用 Conda 环境 |
| 指向系统目录 | 可能不是本项目环境，需要重新选择内核 |

这一步不是背路径，而是确认“Notebook 的 Python”和“安装包的 Python”是同一个。


## 3. 找到项目根目录 (project root)

项目根目录 (project root) 是包含 `pyproject.toml` 和 `src/aurora` 的文件夹。本项目应为 `/Users/z/AURORA`。


In [ ]:
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = None
for folder in candidates:
    if (folder / 'pyproject.toml').exists() and (folder / 'src' / 'aurora').exists():
        project_root = folder
        break

print('project_root =', project_root)
assert project_root is not None, '没有找到项目根目录：请确认从 /Users/z/AURORA 启动 jupyter lab'
assert (project_root / 'src' / 'aurora').exists()


## 4. 检查核心依赖 (core dependencies)

后续课程至少需要这些包：

- NumPy (NumPy)：数组与数值计算
- Matplotlib (Matplotlib)：画图
- Aurora (Aurora package)：本项目源码包
- IPython kernel (IPython kernel)：Notebook 内核


In [ ]:
import importlib

packages = ['numpy', 'matplotlib', 'aurora', 'ipykernel']
for name in packages:
    module = importlib.import_module(name)
    version = getattr(module, '__version__', 'version not exposed')
    print(f'{name:10s} OK | {version}')


## 5. 检查 Aurora 源码是否可访问 (source package check)

可编辑安装 (editable install) 的好处是：Notebook 里导入的 `aurora` 会直接连到 `src/aurora`。修改源码后，不需要重新复制一份包。


In [ ]:
import aurora
from pathlib import Path

print('aurora module file:', Path(aurora.__file__).resolve())
assert 'src/aurora' in str(Path(aurora.__file__).resolve())


## 6. 第一次小实践：生成环境报告 (environment report)

把下面的 TODO 补完。目标是得到一个字典 (dictionary)，里面记录当前 Notebook 的运行环境。


In [ ]:
import sys
from pathlib import Path

# TODO: 把 None 换成对应值。
environment_report = {
    'python_executable': None,
    'python_version': None,
    'project_root': None,
    'kernel_expected': 'Python (AURORA)',
}

environment_report


### 检查：环境报告 (check environment report)

补完上一格后运行这一格。


In [ ]:
assert environment_report['python_executable'] == sys.executable
assert environment_report['python_version'] == sys.version.split()[0]
assert str(environment_report['project_root']).endswith('AURORA')
assert environment_report['kernel_expected'] == 'Python (AURORA)'
print('环境报告通过')


## 7. 推演：如果导入失败，先看哪三处 (debugging path)

| 现象 | 优先检查 |
|---|---|
| `ModuleNotFoundError: No module named 'aurora'` | 当前内核是否是 `Python (AURORA)`；是否运行过 `pip install -e ".[dev,notebooks]"` |
| `No module named 'matplotlib'` | 是否安装了 notebooks 依赖 |
| `sys.executable` 不在 `.venv` | 是否选错了 Jupyter 内核 |
| `project_root = None` | 是否从 `/Users/z/AURORA` 启动了 `jupyter lab` |

推演顺序：先确认内核 (kernel)，再确认 Python 路径 (Python executable)，再确认包安装 (package installation)，最后确认项目路径 (project path)。


## 8. 第二次小实践：写一个导入检查函数 (import check function)

实现 `check_imports(names)`：输入包名列表，返回一个字典，键是包名，值是 `True` 或 `False`。


In [ ]:
import importlib

def check_imports(names):
    # TODO: 返回 {'numpy': True, 'not_a_package': False} 这样的字典。
    result = {}
    for name in names:
        result[name] = None
    return result

check_imports(['numpy', 'matplotlib', 'aurora', 'not_a_real_package'])


### 检查：导入检查函数 (check import function)


In [ ]:
status = check_imports(['numpy', 'matplotlib', 'aurora', 'not_a_real_package'])
assert status['numpy'] is True
assert status['matplotlib'] is True
assert status['aurora'] is True
assert status['not_a_real_package'] is False
print('导入检查函数通过')


## 本课收束 (wrap-up)

本课建立了 5 个基础概念：

1. 虚拟环境 (virtual environment)
2. Jupyter 内核 (Jupyter kernel)
3. Python 解释器 (Python interpreter)
4. 项目根目录 (project root)
5. 可编辑安装 (editable install)

下一课会进入 Notebook 单元格 (notebook cell)、变量 (variable)、NumPy 数组 (NumPy array)、shape 和 assert。
